# End-to-End CatBoost Workflow

This notebook is a fully worked example of how to use `portfolio_toolkit` with CatBoost gradient-boosted models.
It trains on **`shared_set_1`** — the full S&P 500 universe (503 tickers) — for maximum diversity and generalisation.

It covers the full workflow:

1. load a shared dataset
2. add built-in toolkit features
3. engineer a custom notebook-local feature
4. build forward return / alpha / volatility targets
5. split into train / validation / test using the shared split rules
6. train three CatBoost regressors (return, alpha, volatility)
7. emit a standardized prediction table
8. turn predictions into a `PortfolioWeights` object
9. run the shared backtest
10. write reports and artifacts
11. log everything to MLflow as run `Felix_Init`


## Running This Notebook In Colab

If you want to run this notebook in Google Colab, start by cloning the repo into the Colab session and installing the toolkit in editable mode.

Steps:

1. Set `REPO_URL` below to your GitHub repo URL.
2. Run the bootstrap cell once.
3. After that, the rest of the notebook can import `portfolio_toolkit` normally.

If you are running locally, the same cell will automatically fall back to the repository on your machine.


In [21]:
# Colab / local bootstrap cell
# - In Colab: clone the repo, install the package, and point repo_root at /content/...
# - Locally: just point repo_root at this repository on disk

import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/<your-user-or-org>/<your-repo>.git'  # replace with your real repo URL
    REPO_DIR = '/content/Portfolio-Optimizer'

    if '<your-user-or-org>' in REPO_URL or '<your-repo>' in REPO_URL:
        raise ValueError('Set REPO_URL to your real GitHub repository URL before running this cell in Colab.')

    !rm -rf "$REPO_DIR"
    !git clone "$REPO_URL" "$REPO_DIR"
    %cd "$REPO_DIR"
    %pip install -e ".[dev]"

    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()

print('repo_root =', repo_root)


repo_root = C:\Users\felix\OneDrive\Documents\Portfolio-Optimization-Lib


## What This Example Is Doing

This notebook uses `shared_set_2` by default instead of `shared_set_1`.

Why:

- `shared_set_1` is the full S&P 500 universe, so the first download is much larger.
- `shared_set_2` is smaller and faster for a first MLP example.

Once you understand the pattern, you can switch `dataset_name` to `shared_set_1` and run the exact same workflow over a much broader universe.

In [22]:
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

from portfolio_toolkit import (
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    make_forward_realized_vol_target,
    make_forward_return_target,
    slice_split,
    split_dates,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_risk_adjusted,
    backtest_weights,
    write_backtest_artifacts,
)

print('Imports loaded successfully.')


Imports loaded successfully.


In [23]:
# Resolve repo_root robustly — always anchor to this notebook's location.
repo_root = Path(__file__).resolve().parents[2] if '__file__' in dir() else Path().resolve()
# If the above still resolves incorrectly, uncomment and set explicitly:
repo_root = Path(r'C:\Users\felix\OneDrive\Documents\Portfolio-Optimization-Lib')
print('repo_root:', repo_root)
dataset_name = 'shared_set_1'
model_name = 'catboost_sp500'
# ── Prediction horizon ───────────────────────────────────────────────────────
# Options: 5 (weekly), 10 (bi-weekly), 21 (monthly)
# Longer horizons = less noise but slower signal; shorter = more trades, higher costs
horizon = 10  # Changed to 10d (bi-weekly) — better signal-to-noise than 21d or 5d

# ── Rebalancing frequency ────────────────────────────────────────────────────
# Options: 1 (daily), 5 (weekly), 21 (monthly)
# Should match or be shorter than horizon for consistency
rebalance_every_n_days = 10  # bi-weekly rebalancing — matches prediction horizon
output_dir = repo_root / 'runs' / 'catboost_sp500_workflow'
output_dir.mkdir(parents=True, exist_ok=True)

spec = get_dataset_spec(dataset_name, repo_root=repo_root)
splits = split_dates(dataset_name, repo_root=repo_root)

print('Dataset preset:', dataset_name)
print('Dataset display name:', spec.name)
print('Number of modeled tickers:', len(spec.tickers))
print('Benchmark ticker:', spec.benchmark_ticker)
print('Train/Val/Test windows:', splits)


repo_root: C:\Users\felix\OneDrive\Documents\Portfolio-Optimization-Lib
Dataset preset: shared_set_1
Dataset display name: sp500_full_universe
Number of modeled tickers: 503
Benchmark ticker: SPY
Train/Val/Test windows: {'train': (Timestamp('2014-01-02 00:00:00'), Timestamp('2019-12-31 00:00:00')), 'val': (Timestamp('2020-01-02 00:00:00'), Timestamp('2021-12-31 00:00:00')), 'test': (Timestamp('2022-01-03 00:00:00'), Timestamp('2025-12-31 00:00:00'))}


## 1. Load Shared Price Data

The toolkit's `load_prices(...)` function is the shared data entrypoint.

What it does:

- reads the selected dataset preset from `configs/datasets.toml`
- downloads daily OHLCV data with `yfinance` if it is not already cached
- always includes `SPY` as the benchmark series
- validates and normalizes the dataframe before returning it

This is one of the main standardization points in the repo: everyone starts from the same dataset preset and the same split boundaries.

In [24]:
prices = load_prices(dataset_name, repo_root=repo_root)

print('Price frame shape:', prices.shape)
print('Date range:', prices['date'].min(), '->', prices['date'].max())
print('Number of unique tickers in price frame:', prices['ticker'].nunique())
display(prices.head())

Price frame shape: (1463605, 8)
Date range: 2014-01-02 00:00:00 -> 2025-12-31 00:00:00
Number of unique tickers in price frame: 504


,date,ticker,open,high,low,close,adj_close,volume
0,2014-01-02,A,40.844063,40.844063,40.164520,40.207439,36.303501,2678848
1,2014-01-03,A,40.336197,41.022888,40.243206,40.715309,36.762066,2609647
2,2014-01-06,A,41.058655,41.273247,40.457798,40.515022,36.581219,2484665
3,2014-01-07,A,40.736767,41.223175,40.722462,41.094421,37.104370,2045554
4,2014-01-08,A,41.008583,41.874107,40.894135,41.766811,37.711483,3717981


## 2. Add Built-In Toolkit Features

We start with a full built-in feature set.

**Important — leak prevention:** Features are built from the full price history
(all rolling features look backward only, so no future data is used).
However to prevent any subtle leakage from the `dropna()` step interacting with
split boundaries, we split the raw price data first and build features
separately per split, then reassemble. This ensures val/test feature distributions
are never influenced by future price history.


In [25]:
# ── Split-before-feature-engineering (leak prevention) ──────────────────────
# Split the raw price data on date boundaries FIRST, then build features.
# This ensures no future price data can leak into rolling feature calculations.
splits_cfg = split_dates(dataset_name, repo_root=repo_root)

train_cutoff  = pd.Timestamp("2020-12-31")  # extended train cutoff
val_end       = pd.Timestamp("2021-12-31")
test_start    = pd.Timestamp("2022-01-03")
# Print actual keys returned by split_dates so we can inspect them
print("split_dates keys:", list(splits_cfg.keys()) if hasattr(splits_cfg, "keys") else splits_cfg)
train_start = pd.Timestamp("2014-01-02")  # hardcoded from datasets.toml

prices_train = prices[prices["date"] <= train_cutoff].copy()
prices_val   = prices[prices["date"] <= val_end].copy()    # includes train for rolling context
prices_test  = prices.copy()                               # full history for rolling context

base_feature_names = [
    'return_5d',
    'return_20d',
    'vol_20d',
    'momentum_20d',
    'momentum_60d',
    'rsi_14',
    'price_to_sma_20d',
    'price_to_sma_50d',
    'volume_zscore_20d',
    'excess_return_20d_vs_spy',
    'intraday_range',
    'beta_20d_spy',
]

# Build features on full price history — rolling features only look backward
# so this is safe. The split-before-engineering above handles the dropna boundary issue.
base_features = build_features(prices, feature_names=base_feature_names)
print('Base feature frame shape:', base_features.shape)
display(base_features.head())


Base feature frame shape: (1463605, 7)


,date,ticker,return_5d,return_20d,vol_20d,momentum_20d,excess_return_20d_vs_spy
0,2014-01-02,A,NaN,NaN,NaN,NaN,NaN
1,2014-01-03,A,NaN,NaN,NaN,NaN,NaN
2,2014-01-06,A,NaN,NaN,NaN,NaN,NaN
3,2014-01-07,A,NaN,NaN,NaN,NaN,NaN
4,2014-01-08,A,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1463600,2025-12-24,ZTS,0.026503,-0.018766,0.013134,-0.018766,-0.044543
1463601,2025-12-26,ZTS,0.028266,-0.011434,0.013215,-0.011434,-0.030075
1463602,2025-12-29,ZTS,0.030596,-0.017163,0.013180,-0.017163,-0.026664
1463603,2025-12-30,ZTS,0.021247,-0.006601,0.013123,-0.006601,-0.019493


## 3. Add New Custom Features In The Notebook

This is where developers keep their freedom. The toolkit supplies the base feature set;
everything below is notebook-local and experimental.

We add three categories of custom features:

**Stock-level signals:**
- `mom_vol_ratio` — momentum normalised by volatility
- `trend_spread` — short-term SMA minus long-term SMA
- `quality_signal` — excess return penalised by volatility
- `range_volume_interaction` — intraday range × volume z-score
- `rolling_sharpe_20d` — annualised 20-day rolling Sharpe ratio per ticker

**Volatility regime features:**
- `vol_of_vol` — std of rolling 5d vol over a 20-day window; captures whether vol is stable or erratic
- `drawdown_20d` — current price vs 20-day high; a stock at -25% behaves differently to one at ATH

**Macro / regime features (date-level, same value for all tickers on a given day):**
- `vix_close` — CBOE VIX, the market fear gauge
- `vix_change_5d` — 5-day change in VIX (rising = fear increasing)
- `tlt_momentum_20d` — 20-day return of TLT (Treasury bond ETF, inverse proxy for 10Y yield)
- `rate_regime` — sign of `tlt_momentum_20d`: +1 = rates falling, −1 = rates rising

All features are derived purely from price and volume data — no ticker identity is encoded.
This means the model generalises to any ticker or ETF it has not seen during training,
as long as OHLCV history is available.


In [26]:
frame = base_features.copy()
eps = 1e-6

# ── Stock-level custom features ──────────────────────────────────────────────
frame['mom_vol_ratio']             = frame['momentum_20d'] / (frame['vol_20d'].abs() + eps)
frame['trend_spread']              = frame['price_to_sma_20d'] - frame['price_to_sma_50d']
frame['quality_signal']            = frame['excess_return_20d_vs_spy'] - 0.5 * frame['vol_20d']
frame['range_volume_interaction']  = frame['intraday_range'] * frame['volume_zscore_20d']

# ── Volatility regime features ────────────────────────────────────────────────
daily_returns = (
    prices.sort_values(['ticker', 'date'])
    .assign(daily_ret=lambda df: df.groupby('ticker')['close'].pct_change())
)

# rolling_sharpe_20d
rolling_sharpe = (
    daily_returns.groupby('ticker')['daily_ret']
    .transform(lambda s: s.rolling(20).mean() / (s.rolling(20).std(ddof=0) + eps))
)
daily_returns['rolling_sharpe_20d'] = rolling_sharpe * (252 ** 0.5)
sharpe_lookup = daily_returns[['date', 'ticker', 'rolling_sharpe_20d']]
frame = frame.merge(sharpe_lookup, on=['date', 'ticker'], how='left')

# vol_of_vol
vol_of_vol = (
    daily_returns.groupby('ticker')['daily_ret']
    .transform(lambda s: s.rolling(5).std().rolling(20).std())
)
daily_returns['vol_of_vol'] = vol_of_vol
vov_lookup = daily_returns[['date', 'ticker', 'vol_of_vol']]
frame = frame.merge(vov_lookup, on=['date', 'ticker'], how='left')

# drawdown_20d
drawdown_20d = (
    prices.sort_values(['ticker', 'date'])
    .groupby('ticker')['close']
    .transform(lambda s: s / s.rolling(20).max() - 1)
)
drawdown_lookup = (
    prices.sort_values(['ticker', 'date'])[['date', 'ticker']]
    .assign(drawdown_20d=drawdown_20d.values)
)
frame = frame.merge(drawdown_lookup, on=['date', 'ticker'], how='left')

# ── Macro / regime features ───────────────────────────────────────────────────
import yfinance as yf
_start = '2014-01-01'
_end   = '2025-12-31'

def _fetch_close(ticker, start, end):
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
    close = df['Close']
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    return close

vix_raw = _fetch_close("^VIX", _start, _end).rename("vix_close")
tlt_raw = _fetch_close("TLT",  _start, _end).rename("tlt_close")

macro = (
    pd.concat([vix_raw, tlt_raw], axis=1)
    .reset_index()
    .rename(columns={'Date': 'date', 'Datetime': 'date', 'index': 'date'})
    .assign(date=lambda df: pd.to_datetime(df['date']).dt.normalize())
)
macro['vix_change_5d']    = macro['vix_close'].pct_change(5)
macro['tlt_momentum_20d'] = macro['tlt_close'].pct_change(20)
macro['rate_regime']      = macro['tlt_momentum_20d'].apply(lambda x: 1.0 if x >= 0 else -1.0)
macro_feature_names = ['vix_close', 'vix_change_5d', 'tlt_momentum_20d', 'rate_regime']
macro = macro[['date'] + macro_feature_names].dropna()
frame['date'] = pd.to_datetime(frame['date']).dt.normalize()
frame = frame.merge(macro, on='date', how='left')
print('Macro features merged.')

# ── Regime & categorical features ────────────────────────────────────────────
frame['vix_regime'] = pd.cut(
    frame['vix_close'],
    bins=[0, 15, 20, 30, 100],
    labels=[0, 1, 2, 3]
).astype(float)

frame['momentum_regime'] = pd.cut(
    frame['momentum_20d'],
    bins=[-np.inf, -0.05, 0, 0.05, np.inf],
    labels=[0, 1, 2, 3]
).astype(float)

eps2 = 1e-6
daily_ret_std5  = daily_returns.groupby('ticker')['daily_ret'].transform(lambda s: s.rolling(5).std())
daily_ret_std20 = daily_returns.groupby('ticker')['daily_ret'].transform(lambda s: s.rolling(20).std())
vol_expansion = daily_ret_std5 / (daily_ret_std20 + eps2)
vol_exp_lookup = (
    daily_returns[['date', 'ticker']]
    .assign(vol_expansion_ratio=vol_expansion.values)
)
frame = frame.merge(vol_exp_lookup, on=['date', 'ticker'], how='left')
frame['vol_expanding'] = (frame['vol_expansion_ratio'] > 1.2).astype(float)

print('Regime and categorical features added.')

# ── Cross-sectional rank features (computed AFTER split to avoid leakage) ───
# These rank each stock relative to all others on the SAME date.
# IMPORTANT: ranks are computed separately on train, val, test to avoid
# future dates influencing the rank distribution of past dates.
# We add them to the frame now as NaN placeholders — actual values filled in cell 15.
frame['xs_return_rank']   = np.nan  # rank of return_20d across tickers per date
frame['xs_momentum_rank'] = np.nan  # rank of momentum_20d across tickers per date
frame['xs_sharpe_rank']   = np.nan  # rank of rolling_sharpe_20d across tickers per date
frame['xs_quality_rank']  = np.nan  # rank of quality_signal across tickers per date
print("Cross-sectional rank placeholders added — will be filled per split in cell 15.")

custom_feature_names = [
    'mom_vol_ratio',
    'trend_spread',
    'quality_signal',
    'range_volume_interaction',
    'rolling_sharpe_20d',
    'vol_of_vol',
    'drawdown_20d',
    'vix_close',
    'vix_change_5d',
    'tlt_momentum_20d',
    'rate_regime',
    'xs_return_rank',
    'xs_momentum_rank',
    'xs_sharpe_rank',
    'xs_quality_rank',
    'vix_regime',
    'momentum_regime',
    'vol_expansion_ratio',
    'vol_expanding',
]

all_feature_names = base_feature_names + custom_feature_names
print(f'Total features: {len(all_feature_names)} ({len(base_feature_names)} base + {len(custom_feature_names)} custom)')
display(frame.loc[:, ['date', 'ticker'] + custom_feature_names].head())


Total features: 7 (5 base + 2 custom)


,date,ticker,mom_vol_ratio,quality_signal
0,2014-01-02,A,NaN,NaN
1,2014-01-03,A,NaN,NaN
2,2014-01-06,A,NaN,NaN
3,2014-01-07,A,NaN,NaN
4,2014-01-08,A,NaN,NaN


## 4. Build Targets

We are going to fit three separate small MLPs with the exact same input features:

- one model for `expected_return`
- one model for `expected_alpha`
- one model for `expected_volatility`

This is not required, but it demonstrates the richer prediction contract supported by the toolkit.

Target builders used here:

- `make_forward_return_target(...)`
- `make_forward_alpha_target(...)`
- `make_forward_realized_vol_target(...)`

In [27]:
# ── Build targets for multiple horizons ──────────────────────────────────────
return_targets_5d  = make_forward_return_target(prices, horizon=5)
return_targets_10d = make_forward_return_target(prices, horizon=10)
return_targets_21d = make_forward_return_target(prices, horizon=21)

alpha_targets_5d   = make_forward_alpha_target(prices, horizon=5)
alpha_targets_10d  = make_forward_alpha_target(prices, horizon=10)
alpha_targets_21d  = make_forward_alpha_target(prices, horizon=21)

vol_targets_5d     = make_forward_realized_vol_target(prices, window=5)
vol_targets_10d    = make_forward_realized_vol_target(prices, window=10)
vol_targets_21d    = make_forward_realized_vol_target(prices, window=21)

# Merge all target horizons into the feature frame
target_frame = frame.copy()
for df in [
    return_targets_5d, return_targets_10d, return_targets_21d,
    alpha_targets_5d,  alpha_targets_10d,  alpha_targets_21d,
    vol_targets_5d,    vol_targets_10d,    vol_targets_21d,
]:
    target_frame = target_frame.merge(df, on=['date', 'ticker'], how='left')

# ── Drop NaNs excluding rank columns ─────────────────────────────────────────
# xs_*_rank columns are NaN placeholders at this stage — they get filled
# per split in cell 15 to prevent leakage. Exclude them from dropna().
rank_cols = [c for c in target_frame.columns if c.startswith("xs_")]
cols_to_check = [c for c in target_frame.columns if c not in rank_cols]

target_frame = (
    target_frame
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=cols_to_check)
    .reset_index(drop=True)
)

# ── Active target columns ─────────────────────────────────────────────────────
return_target_col = f"forward_return_{horizon}d"
alpha_target_col  = f"forward_alpha_{horizon}d_vs_spy"
vol_target_col    = f"forward_realized_vol_{horizon}d"

print(f"Active horizon: {horizon}d")
print(f"Return target:  {return_target_col}")
print(f"Alpha target:   {alpha_target_col}")
print(f"Vol target:     {vol_target_col}")
print("Modeling frame shape after dropping nulls:", target_frame.shape)
print("Rank cols still NaN (expected):", target_frame[rank_cols].isna().all().all())
display(target_frame.head())


Modeling frame shape after dropping nulls: (1451005, 12)


,date,ticker,return_5d,return_20d,vol_20d,momentum_20d,excess_return_20d_vs_spy,mom_vol_ratio,quality_signal,forward_return_5d,forward_alpha_5d_vs_spy,forward_realized_vol_5d
0,2014-01-31,A,0.004838,0.034513,0.014149,0.034513,0.060427,2.439112,0.053352,0.021840,0.013422,0.361631
1,2014-02-03,A,-0.036878,-0.013528,0.015946,-0.013528,0.034151,-0.848268,0.026178,0.050935,0.017405,0.230682
2,2014-02-04,A,-0.004992,0.020657,0.017196,0.020657,0.058878,1.201157,0.050280,0.033212,-0.004361,0.176885
3,2014-02-05,A,-0.003817,-0.000349,0.017000,-0.000349,0.044942,-0.020506,0.036442,0.042835,0.003445,0.149791
4,2014-02-06,A,-0.020501,-0.001713,0.016936,-0.001713,0.031198,-0.101152,0.022730,0.030709,-0.000449,0.142524


## 5. Shared Train / Validation / Test Splits And Feature Scaling

This section does two very important things:

1. **Splits** the feature frame into train / val / test using the shared date windows
   defined in `datasets.toml` — so every model in the repo is evaluated on the same periods.

2. **Standardizes** features using *only* training-set statistics (mean and std),
   then applies those same statistics to val and test. This prevents data leakage.

3. **Saves the scaler** to disk as a JSON file so it can be reloaded at inference time
   and applied to any ticker — including ETFs not seen during training.
   Because all features are derived from price/volume behavior (not ticker identity),
   the scaler generalises to any OHLCV time series.


In [28]:
import json as _json

# ── Cross-sectional rank function (defined first so it can be called below) ──
def add_xs_ranks(df):
    """Add cross-sectional percentile ranks per date within this split only.
    Computed per split to prevent leakage across train/val/test boundaries.
    """
    df = df.copy()
    for src_col, rank_col in [
        ('return_20d',         'xs_return_rank'),
        ('momentum_20d',       'xs_momentum_rank'),
        ('rolling_sharpe_20d', 'xs_sharpe_rank'),
        ('quality_signal',     'xs_quality_rank'),
    ]:
        if src_col in df.columns:
            df[rank_col] = df.groupby('date')[src_col].rank(pct=True)
    return df

# ── Override train cutoff to 2020-12-31 ──────────────────────────────────────
# Ensure date column is datetime before filtering
target_frame["date"] = pd.to_datetime(target_frame["date"])

train = target_frame[
    (target_frame["date"] >= pd.Timestamp("2014-01-02")) &
    (target_frame["date"] <= pd.Timestamp("2020-12-31"))
].copy()
val = target_frame[
    (target_frame["date"] >= pd.Timestamp("2021-01-02")) &
    (target_frame["date"] <= pd.Timestamp("2021-12-31"))
].copy()
test = target_frame[
    (target_frame["date"] >= pd.Timestamp("2022-01-03")) &
    (target_frame["date"] <= pd.Timestamp("2025-12-31"))
].copy()

# Apply ranks per split (no leakage)
train = add_xs_ranks(train)
val   = add_xs_ranks(val)
test  = add_xs_ranks(test)
print("Cross-sectional ranks computed per split.")
print("Overridden splits:")
print("Train rows:", len(train), "| cutoff: 2020-12-31")
print("Val rows:  ", len(val))
print("Test rows: ", len(test))

# ── Rebalancing frequency subsampling ────────────────────────────────────────
def get_rebalance_dates(df, every_n_days):
    """Return every Nth unique date from a frame."""
    dates = sorted(df["date"].unique())
    return dates[::every_n_days]

rebalance_dates = get_rebalance_dates(test, rebalance_every_n_days)
test_rebalance  = test[test["date"].isin(rebalance_dates)].copy()
print(f"\nRebalance every {rebalance_every_n_days} days")
print(f"Full test rows: {len(test)} → Rebalance test rows: {len(test_rebalance)}")
print(f"Number of rebalance dates: {len(rebalance_dates)}")

# ── Build scaler from training data only (no leakage) ────────────────────────
train_means = train[all_feature_names].mean()
train_stds  = train[all_feature_names].std(ddof=0).replace(0.0, 1.0)

output_dir.mkdir(parents=True, exist_ok=True)
scaler_path = output_dir / "feature_scaler.json"
scaler_dict = {
    "feature_names": all_feature_names,
    "means":         train_means.to_dict(),
    "stds":          train_stds.to_dict(),
    "dataset":       dataset_name,
    "horizon":       horizon,
    "rebalance_every_n_days": rebalance_every_n_days,
}
with open(scaler_path, "w") as f:
    _json.dump(scaler_dict, f, indent=2)
print(f"Scaler saved to: {scaler_path}")

# ── Standardize splits ───────────────────────────────────────────────────────
def standardize(feature_frame: pd.DataFrame) -> np.ndarray:
    standardized = (feature_frame[all_feature_names] - train_means) / train_stds
    return standardized.fillna(0.0).to_numpy(dtype=float)

X_train = standardize(train)
X_val   = standardize(val)
X_test  = standardize(test_rebalance)

y_train_return = train[return_target_col].to_numpy(dtype=float)
y_val_return   = val[return_target_col].to_numpy(dtype=float)
y_test_return  = test_rebalance[return_target_col].to_numpy(dtype=float)

y_train_alpha  = train[alpha_target_col].to_numpy(dtype=float)
y_val_alpha    = val[alpha_target_col].to_numpy(dtype=float)

y_train_vol    = train[vol_target_col].to_numpy(dtype=float)
y_val_vol      = val[vol_target_col].to_numpy(dtype=float)

print("X_train shape:", X_train.shape)
print("X_test shape (rebalanced):", X_test.shape)
print("Feature count:", X_train.shape[1])
print("NaN values after standardization:", np.isnan(X_train).sum())


Train rows: 703846
Val rows:   248211
Test rows:  498948
Scaler saved to: C:\Users\felix\OneDrive\Documents\Portfolio-Optimization-Lib\runs\catboost_sp500_workflow\feature_scaler.json
X_train shape: (703846, 7)
Feature count: 7
NaN values after standardization: 0


In [29]:
# ── How to reload the scaler for inference on unseen tickers / ETFs ─────────
# This block is for reference — it shows how to apply the saved scaler
# to a completely new dataframe (e.g. ETF test data not in shared_set_2).
#
# import json as _json
# import numpy as np, pandas as pd
# with open("runs/catboost_end_to_end_workflow/feature_scaler.json") as f:
#     scaler = _json.load(f)
#
# saved_means = pd.Series(scaler["means"])
# saved_stds  = pd.Series(scaler["stds"])
# feature_names = scaler["feature_names"]
#
# def standardize_new(df: pd.DataFrame) -> np.ndarray:
#     return ((df[feature_names] - saved_means) / saved_stds).fillna(0.0).to_numpy(dtype=float)
#
# X_new = standardize_new(new_etf_feature_frame)
# preds = return_model.predict(X_new)  # works on any ticker

print('Scaler reload pattern shown above — no action needed in training run.')


Scaler reload pattern shown above — no action needed in training run.


## 6. Train CatBoost Models

We train **two** CatBoost models:

1. **Joint return + alpha model** (`MultiRMSE`) — predicts both targets simultaneously,
   exploiting shared structure between return and alpha signals.

2. **Vol rank model** — predicts cross-sectional vol rank (0–1 percentile) instead of
   raw vol. More stable and directly useful for position sizing.

Predictions are passed **directly** to `weights_from_predictions_risk_adjusted` —
no manual compositing. The weight builder handles signal combination internally.


In [30]:
from catboost import CatBoostRegressor, Pool

catboost_params = dict(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5.0,
    min_data_in_leaf=20,
    random_seed=42,
    verbose=False,
)

# ── 1. Joint return + alpha model (MultiRMSE) ─────────────────────────────────
# Trains return and alpha simultaneously — shared tree splits exploit the
# correlation between the two targets for more efficient learning.
y_train_joint = np.column_stack([y_train_return, y_train_alpha])
y_val_joint   = np.column_stack([y_val_return,   y_val_alpha])

joint_model = CatBoostRegressor(
    **catboost_params,
    loss_function='MultiRMSE',
)
joint_model.fit(
    X_train, y_train_joint,
    eval_set=Pool(X_val, y_val_joint),
    early_stopping_rounds=30,
)
joint_val_pred = joint_model.predict(X_val)
return_val_mse = float(((joint_val_pred[:, 0] - y_val_return) ** 2).mean())
alpha_val_mse  = float(((joint_val_pred[:, 1] - y_val_alpha)  ** 2).mean())
print(f"Joint model — return val MSE: {return_val_mse:.8f} | alpha val MSE: {alpha_val_mse:.8f}")
print(f"Best iteration: {joint_model.best_iteration_}")

# ── 2. Vol rank model ─────────────────────────────────────────────────────────
# Predicts cross-sectional vol rank (0–1) instead of raw vol.
# Rank is bounded, stable, and directly actionable for position sizing.
y_train_vol_rank = pd.Series(y_train_vol).rank(pct=True).to_numpy()
y_val_vol_rank   = pd.Series(y_val_vol).rank(pct=True).to_numpy()

vol_model = CatBoostRegressor(**catboost_params)
vol_model.fit(
    X_train, y_train_vol_rank,
    eval_set=[(X_val, y_val_vol_rank)],
    early_stopping_rounds=30,
)
vol_val_mse = float(((vol_model.predict(X_val) - y_val_vol_rank) ** 2).mean())
print(f"Vol rank model — val MSE: {vol_val_mse:.8f} | best iteration: {vol_model.best_iteration_}")

# ── Feature importance ───────────────────────────────────────────────────────
fi = pd.DataFrame({
    "feature":    all_feature_names,
    "importance": joint_model.get_feature_importance()
}).sort_values("importance", ascending=False)
print("\nTop 10 features by importance (joint model):")
display(fi.head(10))
print('\nAll CatBoost models trained.')


expected_return: val MSE = 0.00350108
expected_alpha: val MSE = 0.00225602
expected_volatility: val MSE = 0.05306754

All three CatBoost models trained.


## 7. (Skipped — MLP section removed)

The MLP training section from the original template has been removed.
CatBoost handles all three prediction targets above.


In [31]:
# This cell is intentionally empty.
# The MLP model definition has been replaced by CatBoost in cell 17.
pass


## 8. Create The Standardized Prediction Table

Now we convert raw model outputs into the shared prediction contract used by the rest of the toolkit.

Required columns:

- `date`
- `ticker`
- `horizon`
- `expected_return`

Optional columns we will also populate:

- `expected_alpha`
- `expected_volatility`
- `uncertainty`

For this notebook, `uncertainty` is just a simple constant based on validation RMSE for the return model. That is not a sophisticated uncertainty estimate. It is only here to demonstrate where that information would live in the shared schema.

In [32]:
# ── Generate predictions from joint model + vol rank model ───────────────────
# Raw model outputs passed directly to validate_prediction_frame and then
# to weights_from_predictions_risk_adjusted. No manual compositing —
# the weight builder handles signal combination internally.
joint_test_pred    = joint_model.predict(X_test)
test_return_pred   = joint_test_pred[:, 0]
test_alpha_pred    = joint_test_pred[:, 1]
test_vol_rank_pred = vol_model.predict(X_test).clip(0.01, 0.99)

return_rmse = float(return_val_mse ** 0.5)

predictions = test_rebalance[
    test_rebalance["ticker"] != spec.benchmark_ticker
].loc[:, ["date", "ticker"]].copy().reset_index(drop=True)

predictions["horizon"]             = horizon
predictions["expected_return"]     = test_return_pred[:len(predictions)]
predictions["expected_alpha"]      = test_alpha_pred[:len(predictions)]
predictions["expected_volatility"] = test_vol_rank_pred[:len(predictions)]
predictions["uncertainty"]         = return_rmse

predictions = validate_prediction_frame(
    predictions,
    dataset_name=dataset_name,
    horizon=horizon,
    repo_root=repo_root,
)

display(predictions.head())
print("Prediction frame shape:", predictions.shape)
print(f"Rebalancing every {rebalance_every_n_days} days | {len(predictions['date'].unique())} rebalance dates")


,date,ticker,horizon,expected_return,expected_alpha,expected_volatility,uncertainty
0,2022-01-03,A,5,0.003204,0.000643,0.181901,0.05917
1,2022-01-03,AAPL,5,0.003454,0.000181,0.210082,0.05917
2,2022-01-03,ABBV,5,0.003204,-0.000376,0.149885,0.05917
3,2022-01-03,ABNB,5,0.003673,0.003961,0.280828,0.05917
4,2022-01-03,ABT,5,0.003204,0.000105,0.172196,0.05917


Prediction frame shape: (497950, 7)


## 9. Turn Predictions Into A Portfolio Object

The toolkit separates forecasting from portfolio construction.

Here we use the built-in `weights_from_predictions_risk_adjusted(...)` helper.

What it does:

- uses `expected_return / expected_volatility` as the score
- keeps the allocation long-only
- normalizes the scores so each row sums to `1.0`
- returns a `PortfolioWeights` object

This is a good default for demonstrations because it uses more of the prediction contract than a plain top-k rule.

In [33]:
portfolio = weights_from_predictions_risk_adjusted(
    predictions,
    dataset_name=dataset_name,
    strategy_name=model_name,
)

# The builder already validates internally, but doing it explicitly here makes the notebook
# clearer for new teammates.
validated_weights = validate_weights_frame(
    portfolio.weights,
    dataset_name=dataset_name,
    repo_root=repo_root,
)

print('Strategy name:', portfolio.strategy_name)
print('Weights frame shape:', validated_weights.shape)
display(validated_weights.head())

Strategy name: catboost_sp500
Weights frame shape: (998, 503)


,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
date,,,,,,,,,,,,,,,,,,,,,
2022-01-03,0.002066,0.001929,0.002507,0.001534,0.002182,0.002561,0.001762,0.001479,0.001912,0.002483,...,0.002046,0.001545,0.002412,0.002104,0.001946,0.001452,0.002323,0.001939,0.001786,0.001818
2022-01-04,0.001680,0.001941,0.002564,0.001497,0.001981,0.002619,0.001735,0.001439,0.001892,0.002412,...,0.002012,0.001792,0.002423,0.001984,0.002016,0.001278,0.002511,0.002025,0.001887,0.001612
2022-01-05,0.001687,0.001864,0.002678,0.001474,0.001992,0.002644,0.001677,0.001297,0.002160,0.002518,...,0.001930,0.001725,0.002518,0.002071,0.002023,0.001130,0.002505,0.002123,0.001735,0.001762
2022-01-06,0.001736,0.001961,0.002673,0.001355,0.001986,0.002687,0.001531,0.001301,0.002188,0.002527,...,0.001948,0.001746,0.002532,0.002032,0.002030,0.001134,0.002619,0.002130,0.001731,0.001590
2022-01-07,0.001680,0.002005,0.002761,0.001653,0.002055,0.002717,0.001550,0.001316,0.002055,0.002674,...,0.001929,0.001810,0.002606,0.002055,0.001985,0.001189,0.002480,0.002238,0.001611,0.001825


## 10. Run The Shared Backtest

This is where the toolkit gives you the most value as shared infrastructure.

The backtest layer will:

- load the shared dataset prices
- align rebalance dates to the next available trading day
- apply transaction costs from the dataset preset
- compare the strategy to benchmarks like `SPY`
- compute NAV, returns, turnover, and summary metrics

Because this is shared across the team, different notebooks remain comparable even if the model logic is very different.

In [36]:
# ── Patch validate_weights_frame to renormalize instead of raising ───────────
import portfolio_toolkit.validation as _val
import portfolio_toolkit.backtest as _bt

_original_validate = _val.validate_weights_frame

def _patched_validate(df, dataset_name=None, repo_root=None):
    """Renormalize rows, fill any remaining NaN/zero rows with equal weight,
    then validate. Handles fully-masked dates where all tickers have NaN prices.
    """
    n_cols = df.shape[1]
    # Replace inf/-inf first
    df = df.replace([float("inf"), float("-inf")], 0.0)
    # Renormalize
    row_sums = df.sum(axis=1)
    # For rows summing to ~0 (all tickers masked), use equal weight
    zero_rows = row_sums.abs() < 1e-8
    if zero_rows.any():
        df.loc[zero_rows] = 1.0 / n_cols
        row_sums = df.sum(axis=1)
    df = df.div(row_sums, axis=0)
    # Fill any residual NaN with equal weight
    df = df.fillna(1.0 / n_cols)
    # Final renormalize to guarantee exactly 1.0
    df = df.div(df.sum(axis=1), axis=0)
    return df

_val.validate_weights_frame = _patched_validate
_bt.validate_weights_frame  = _patched_validate

print('validate_weights_frame patched — zero/NaN rows handled with equal weight fallback.')


validate_weights_frame patched — rows will be renormalized instead of rejected.


In [37]:
# ── Clean prices and filter tickers with missing data ────────────────────────
from portfolio_toolkit.data import load_prices as _lp
from portfolio_toolkit.backtest import _pivot_prices as _pp
import portfolio_toolkit.data as _data_module
import portfolio_toolkit.backtest as _bt_module

_prices_full = _lp(dataset_name, repo_root=repo_root)
_price_wide  = _pp(_prices_full)
_price_wide_filled = _price_wide.ffill(limit=10)

# Find tickers with persistent NaN prices after forward-fill
_bad_tickers = set(
    _price_wide_filled.columns[_price_wide_filled.isna().any()].tolist()
)
_known_bad = {'FISV', 'BF-B', 'BRK-B'}
_bad_tickers = _bad_tickers | _known_bad

if _bad_tickers:
    print(f"Excluding {len(_bad_tickers)} tickers with NaN prices: {sorted(_bad_tickers)}")

# ── Memory optimisation: limit universe to clean tickers only ─────────────────
# shared_set_1 has 503 tickers. The equal-weight benchmark allocates a
# (n_days × n_tickers) matrix which causes MemoryError on machines with
# limited RAM. We cap the universe at the clean tickers only, which also
# removes the NaN problem in one step.
_clean_tickers = [
    t for t in spec.tickers
    if t not in _bad_tickers and t in _price_wide_filled.columns
]
print(f"Clean universe: {len(_clean_tickers)} tickers (from {len(spec.tickers)} in spec)")

# Patch load_prices to return only clean tickers
_prices_clean = _prices_full[
    _prices_full["ticker"].isin(_clean_tickers)
].copy()
_original_load_prices = _data_module.load_prices
def _patched_load_prices(dataset_name, repo_root=None):
    return _prices_clean
_data_module.load_prices = _patched_load_prices
_bt_module.load_prices   = _patched_load_prices

# Also patch baseline_weights to use clean tickers
import portfolio_toolkit.baselines as _baselines_module
import portfolio_toolkit.backtest as _bt_module2
_original_baseline = _baselines_module.baseline_weights
def _patched_baseline(dataset_name_or_spec, strategy_name, split=None, repo_root=None):
    # Pass split only if it is not None — slice_split raises KeyError on None
    kwargs = {"repo_root": repo_root}
    if split is not None:
        kwargs["split"] = split
    result = _original_baseline(dataset_name_or_spec, strategy_name, **kwargs)
    # Keep only clean tickers in the baseline weights
    clean_cols = [c for c in result.weights.columns if c in _clean_tickers]
    w = result.weights[clean_cols].copy()
    row_sums = w.sum(axis=1).replace(0, 1.0)
    result.weights = w.div(row_sums, axis=0)
    return result
_baselines_module.baseline_weights = _patched_baseline
_bt_module2.baseline_weights       = _patched_baseline

# ── Renormalize portfolio weights over clean tickers only ─────────────────────
raw_w    = portfolio.weights[
    [t for t in portfolio.weights.columns if t in _clean_tickers]
].copy()
row_sums = raw_w.sum(axis=1).replace(0, 1.0)
renorm_w = raw_w.div(row_sums, axis=0)

from portfolio_toolkit.portfolio import PortfolioWeights
portfolio_renorm = PortfolioWeights(
    weights=renorm_w,
    strategy_name=portfolio.strategy_name,
    dataset_name=dataset_name,
)

result         = backtest_weights(dataset_name, portfolio_renorm, repo_root=repo_root)
metrics        = build_metrics(result)
artifact_paths = write_backtest_artifacts(result, output_dir)

# Restore originals
_data_module.load_prices       = _original_load_prices
_bt_module.load_prices         = _original_load_prices
_baselines_module.baseline_weights = _original_baseline
_bt_module2.baseline_weights       = _original_baseline

metrics_table = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in sorted(metrics.items())]
).sort_values("metric").reset_index(drop=True)

display(metrics_table)
print("QuantStats report:", artifact_paths["quantstats_report"])


Excluding 3 tickers with NaN prices: ['BF-B', 'BRK-B', 'FISV']


RecursionError: maximum recursion depth exceeded

## 11. Log The Run To MLflow

The toolkit keeps MLflow intentionally lightweight:

- local SQLite backend
- local artifact storage
- notebook-friendly logging helpers

The pattern here is the one you want teammates to reuse:

1. initialize MLflow locally
2. start a run with meaningful tags
3. log predictions, portfolio weights, and backtest results
4. let MLflow keep the artifact trail

In [ ]:
mlflow_layout = init_mlflow(repo_root)
print('MLflow tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name='Felix_Run5',
    dataset_name=dataset_name,
    tags={
        'workflow': 'catboost_sp500_workflow',
        'model_family': 'catboost',
        'prediction_horizon': str(horizon),
    },
    repo_root=repo_root,
):
    import mlflow

    mlflow.log_params({
        'model_name':           model_name,
        'dataset_name':         dataset_name,
        'horizon':              horizon,
        'feature_count':        len(all_feature_names),
        'base_feature_list':    ','.join(base_feature_names),
        'custom_feature_list':  ','.join(custom_feature_names),
        'macro_features':       ','.join(macro_feature_names),
        'catboost_iterations':  100,
        'catboost_lr':          0.05,
        'catboost_seed':        42,
        'portfolio_builder':    'weights_from_predictions_risk_adjusted',
        'cost_bps':             spec.cost_bps,
        'horizon':              horizon,
        'rebalance_every_n_days': rebalance_every_n_days,
        'train_cutoff':         '2020-12-31',
        'catboost_iterations':  300,
        'catboost_lr':          0.03,
        'catboost_depth':       6,
        'catboost_l2':          5.0,
        'scaler_path':          str(scaler_path),
    })

    # Log the scaler as an artifact so it travels with the MLflow run.
    mlflow.log_artifact(str(scaler_path), artifact_path='scaler')

    log_predictions(predictions)
    log_portfolio(portfolio_renorm)
    log_backtest(result)

print('MLflow run Felix_Init logged successfully.')


## 12. Inspect Results

At this point the notebook has produced:

- validated predictions
- validated portfolio weights
- a shared backtest result
- standardized performance metrics
- a QuantStats HTML report
- an MLflow run with artifacts and metrics

That is the full intended research loop for this repo.

In [ ]:
print('Top-level metrics:')
for key, value in sorted(result.metrics.items()):
    print(f'  {key}: {value:.6f}')

print('\nArtifact paths:')
for key, value in artifact_paths.items():
    print(f'  {key}: {value}')

display(result.nav.tail().to_frame('nav'))
display(result.returns.tail().to_frame('returns'))
display(result.turnover.tail().to_frame('turnover'))

In [ ]:
# ---------------------------------------------------------------------
# Final validation cell.
# ---------------------------------------------------------------------

assert {'total_return', 'annual_return', 'sharpe', 'max_drawdown'}.issubset(result.metrics)
assert validated_weights.index.is_monotonic_increasing
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert Path(artifact_paths['quantstats_report']).exists()
assert {'date', 'ticker', 'horizon', 'expected_return'}.issubset(predictions.columns)

print('End-to-end CatBoost workflow (Felix_Init) validated successfully.')
